In [2]:
# =================================================================================================
# Raw data : This is Questionnaire data from NHANES 2003-2004
# Purpose : Creating unified raw CSV file from NHANES questionnaire CSV files.
# Specification : Each NHANES dataset has SEQN "Participant ID", which is the unique respondent ID.
# Files Used : ALQ_C.csv, BPQ_C.csv, DIQ_C.csv, HIQ_C.csv, HUQ_C.csv, SMQ_C.csv, SMQFAM_C.csv, WHQ_C.csv
# Data Dictionary : CDC NHANES 2003-2004 Questionnaire Data Dictionary
# Data Availability Matrix : Created in this notebook using SEQN presence flags
# Source Site : https://wwwn.cdc.gov/nchs/nhanes/search/datapage.aspx?Component=Questionnaire&CycleBeginYear=2003
# Developer Notes : The notebook was created for Windows desktop environment with Python3.
# File Creator : Neha Khairnar
# =================================================================================================
# 📦 Step 1: Imports
import pandas as pd
from pathlib import Path

# =================================================================================================
# 📂 Step 2: Paths below resolved for locally run datasets
# =================================================================================================

BASE_DIR = Path().resolve().parent
DATA_DIR = BASE_DIR / "data" / "2011_2012" / "Questionnnair"

alcohol_file = DATA_DIR / "ALQ_G_corrected.csv"
bp_chol_file = DATA_DIR / "BPQ_G_corrected.csv"
diabetes_file = DATA_DIR / "DIQ_G_corrected.csv"
insurance_file = DATA_DIR / "HIQ_G_corrected.csv"
access_file = DATA_DIR / "HUQ_G_corrected.csv"
smoking_file = DATA_DIR / "SMQ_G_corrected.csv"
household_smokers_file = DATA_DIR / "SMQFAM_G_corrected.csv"
weight_history_file = DATA_DIR / "WHQ_G_corrected.csv"
Housing_file = DATA_DIR / "HOQ_G_corrected.csv"
Occupation_file = DATA_DIR / "OCQ_G_corrected.csv"
medical_cndition_file = DATA_DIR / "MCQ_G_corrected.csv"
physical_file = DATA_DIR / "PAQ_G_corrected.csv"
#Osteo_file = DATA_DIR / "OSQ_G_corrected.csv"
Sleep_file = DATA_DIR / "SLQ_G_corrected.csv"

files = {
    "alcohol_ALQ_C": alcohol_file,
    "blood_pressure_cholesterol_BPQ_C": bp_chol_file,
    "diabetes_DIQ_C": diabetes_file,
    "health_insurance_HIQ_C": insurance_file,
    "hospital_access_HUQ_C": access_file,
    "smoking_SMQ_C": smoking_file,
    "household_smokers_SMQFAM_C": household_smokers_file,
    "weight_history_WHQ_C": weight_history_file,    
    "Housing_file_HOQ_C" : Housing_file,
    "Occupation_file_OCQ_C":Occupation_file,
    "medical_cndition_file_MCQ_C":medical_cndition_file,
    "physical_file" :physical_file,
    #"Osteo_file" : Osteo_file,
    "Sleep_file" : Sleep_file
}

# =================================================================================================
# 📥 Step 3: Load ALL Files
# =================================================================================================

dfs = {}
for name, path in files.items():
    df_temp = pd.read_csv(path, dtype=str,keep_default_na=False)
    df_temp.columns = df_temp.columns.str.upper()
    dfs[name] = df_temp

summary = []
for name, df_temp in dfs.items():
    summary.append({
        "file": name,
        "rows": df_temp.shape[0],
        "columns": df_temp.shape[1],
        "column_names": list(df_temp.columns)
    })

summary_df = pd.DataFrame(summary)
print("📊 NHANES Questionnaire File Structure Summary:\n")
display(summary_df[["file", "rows", "columns", "column_names"]])
for i, row in summary_df.iterrows():
    print("\n" + "="*80)
    print("FILE:", row["file"])
    print("COLUMNS:", len(row["column_names"]))
    print(row["column_names"])

# =================================================================================================
# 🔗 Step 5: Merge ALL columns using FULL OUTER JOIN on SEQN
# =================================================================================================

df = dfs["alcohol_ALQ_C"]
for name in [
    "blood_pressure_cholesterol_BPQ_C",
    "diabetes_DIQ_C",
    "health_insurance_HIQ_C",
    "hospital_access_HUQ_C",
    "smoking_SMQ_C",
    "household_smokers_SMQFAM_C",
    "weight_history_WHQ_C",    
    "Housing_file_HOQ_C",
    "Occupation_file_OCQ_C",
    "medical_cndition_file_MCQ_C",
    "physical_file",
    #"Osteo_file",
    "Sleep_file"
]:
    df = df.merge(
        dfs[name],
        on="SEQN",
        how="outer",
        suffixes=("", f"_{name}")
    )

# =================================================================================================
# 💾 Step 6: Save Final Output
# =================================================================================================
print("📋 Final Columns After Merge + Rename:")
print(df.columns.tolist())

print("\n📊 Final Shape:", df.shape)
display(df.head(20))


OUTPUT_DIR = BASE_DIR / "data" / "processed_v3" / "2011_2012"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_PATH = OUTPUT_DIR / "neighbourhood_nhanes_questionnaire_2011_2012_raw_corrected.csv"
df.to_csv(OUTPUT_PATH, index=False)
print("\n✅ Saved to:")
print(OUTPUT_PATH)



📊 NHANES Questionnaire File Structure Summary:



,file,rows,columns,column_names
0,alcohol_ALQ_C,5615,10,"[SEQN, ALQ101, ALQ110, ALQ120Q, ALQ120U, ALQ13..."
1,blood_pressure_cholesterol_BPQ_C,6175,15,"[SEQN, BPQ020, BPQ030, BPD035, BPQ040A, BPQ050..."
2,diabetes_DIQ_C,9364,53,"[SEQN, DIQ010, DID040, DIQ160, DIQ170, DIQ172,..."
3,health_insurance_HIQ_C,9756,17,"[SEQN, HIQ011, HIQ031A, HIQ031B, HIQ031C, HIQ0..."
4,hospital_access_HUQ_C,9756,10,"[SEQN, HUQ010, HUQ020, HUQ030, HUQ040, HUQ050,..."
5,smoking_SMQ_C,6790,30,"[SEQN, SMQ020, SMD030, SMQ040, SMQ050Q, SMQ050..."
6,household_smokers_SMQFAM_C,9756,5,"[SEQN, SMD410, SMD415, SMD415A, SMD430]"
7,weight_history_WHQ_C,6175,63,"[SEQN, WHD010, WHD020, WHQ030, WHQ040, WHD045,..."
8,Housing_file_HOQ_C,9756,3,"[SEQN, HOD050, HOQ065]"
9,Occupation_file_OCQ_C,6175,22,"[SEQN, OCD150, OCQ180, OCQ210, OCD231, OCD241,..."



FILE: alcohol_ALQ_C
COLUMNS: 10
['SEQN', 'ALQ101', 'ALQ110', 'ALQ120Q', 'ALQ120U', 'ALQ130', 'ALQ141Q', 'ALQ141U', 'ALQ151', 'ALQ155']

FILE: blood_pressure_cholesterol_BPQ_C
COLUMNS: 15
['SEQN', 'BPQ020', 'BPQ030', 'BPD035', 'BPQ040A', 'BPQ050A', 'BPQ057', 'BPQ056', 'BPD058', 'BPQ059', 'BPQ080', 'BPQ060', 'BPQ070', 'BPQ090D', 'BPQ100D']

FILE: diabetes_DIQ_C
COLUMNS: 53
['SEQN', 'DIQ010', 'DID040', 'DIQ160', 'DIQ170', 'DIQ172', 'DIQ175A', 'DIQ175B', 'DIQ175C', 'DIQ175D', 'DIQ175E', 'DIQ175F', 'DIQ175G', 'DIQ175H', 'DIQ175I', 'DIQ175J', 'DIQ175K', 'DIQ175L', 'DIQ175M', 'DIQ175N', 'DIQ175O', 'DIQ175P', 'DIQ175Q', 'DIQ175R', 'DIQ175S', 'DIQ175T', 'DIQ175U', 'DIQ175V', 'DIQ175W', 'DIQ180', 'DIQ050', 'DID060', 'DIQ060U', 'DIQ070', 'DIQ230', 'DIQ240', 'DID250', 'DID260', 'DIQ260U', 'DIQ275', 'DIQ280', 'DIQ291', 'DIQ300S', 'DIQ300D', 'DID310S', 'DID310D', 'DID320', 'DID330', 'DID341', 'DID350', 'DIQ350U', 'DIQ360', 'DIQ080']

FILE: health_insurance_HIQ_C
COLUMNS: 17
['SEQN', 'HIQ011', 'HIQ0

,SEQN,ALQ101,ALQ110,ALQ120Q,ALQ120U,ALQ130,ALQ141Q,ALQ141U,ALQ151,ALQ155,...,PAQ665,PAQ670,PAD675,PAD680,PAQ710,PAQ715,PAAQUEX,SLD010H,SLQ050,SLQ060
0,62161,2,2,,,,,,,,...,2,,,300,3,3,1,8,2,2
1,62162,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,,,,,0,8,1,NaN,NaN,NaN
2,62163,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,2,,,720,2,3,2,NaN,NaN,NaN
3,62164,,,,,,,,,,...,1,1,45,300,2,8,1,8,1,2
4,62165,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,1,2,25,600,5,2,2,NaN,NaN,NaN
5,62166,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,,,,,2,0,1,NaN,NaN,NaN
6,62167,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,62168,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,,,,,3,2,1,NaN,NaN,NaN
8,62169,1,,1,2,2,1,3,2,,...,2,,,360,3,2,1,8,2,2
9,62170,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,2,,,600,1,2,2,NaN,NaN,NaN



✅ Saved to:
C:\Users\User\Desktop\Projects\Clinical_Neighbourhood_ML_Data_Predictor\Clinical_Neighbourhood_ML_Data_Predictor\data\processed_v3\2011_2012\neighbourhood_nhanes_questionnaire_2011_2012_raw_corrected.csv
